# nb_29_identity_eval

(This is only a test notebook. Not part of the pipeline) It measures how good the identity matching in `nb_22` is.

The generator wrote an answer key (`person_truth.csv`) that says which client records belong to the same person. This notebook compares the key with what `silver_client_identity` decided.

The answer key lives in its own folder (`Files/eval/`). The pipeline never reads it. A real company would not have such a file, so the pipeline must not depend on it.

Measures
- **Recall**: of the people who really have two records, how many did we link?
- **Precision**: of the links we made, how many were right?
- **False merges**: two different people put together (the costly mistake, because it can mix up consent and purchase history).
- The same numbers split by why a match was hard (typo, nickname, different email, ...).

In [ ]:
# ---- Settings -------------------------------------------------------------------------
TRUTH_PATH = "Files/eval/person_truth.csv"

In [ ]:
import uuid
from datetime import datetime, timezone
import pyspark.sql.functions as F

truth = (spark.read.option("header", True).csv(TRUTH_PATH)
         .select("person_id", "client_id", "record_type", "name_variant",
                 F.col("email_same_as_primary").alias("email_same"), F.col("phone_same_as_primary").alias("phone_same")))
ident = spark.table("silver_client_identity").select("client_id", "person_key", "link_rule", "link_confidence")
print("answer key rows:", truth.count(), "| identity map rows:", ident.count())

# Join the key to what the pipeline decided. Records that never reached silver (rejected in nb_20) cannot be matched.
t = truth.join(ident, "client_id", "left").withColumn("in_silver", F.col("person_key").isNotNull())
t.groupBy("in_silver").count().show()

In [ ]:
# ---- Recall: real two-record people ---------------------------------------------------
two = t.groupBy("person_id").agg(F.count("*").alias("records"), F.sum(F.col("in_silver").cast("int")).alias("in_silver"),
                                 F.countDistinct("person_key").alias("keys")).filter("records = 2")
findable = two.filter("in_silver = 2")
found = findable.filter("keys = 1")
print("people with two records in the key:", two.count())
print("  both records reached silver (findable):", findable.count())
print("  linked to one person_key (found):", found.count())
print(f"  RECALL = {found.count() / max(findable.count(), 1):.3f}")

In [ ]:
# ---- Precision and false merges -------------------------------------------------------
# Group by person_key; a key that holds records of 2+ real people is a false merge.
groups = t.filter("in_silver").groupBy("person_key").agg(F.count("*").alias("records"), F.countDistinct("person_id").alias("real_people"))
merged = groups.filter("records > 1")
false_merges = merged.filter("real_people > 1").count()
print("person_keys with 2+ records:", merged.count())
print("  of which hold two different real people (false merges):", false_merges)
print(f"  PRECISION = {1 - false_merges / max(merged.count(), 1):.3f}")

In [ ]:
# ---- Where does it miss? Recall by reason ---------------------------------------------
sec = t.filter("record_type = 'secondary'").drop("in_silver").join(two.select("person_id", "keys", "in_silver"), "person_id", "left")
sec = sec.withColumn("found", (F.col("in_silver") == 2) & (F.col("keys") == 1))
(sec.filter("in_silver = 2")
    .withColumn("what_matched", F.concat(F.when(F.col("email_same") == "True", "same email").otherwise("diff email"), F.lit(" + "),
                                         F.when(F.col("phone_same") == "True", "same phone").otherwise("diff phone")))
    .groupBy("what_matched").agg(F.count("*").alias("people"), F.sum(F.col("found").cast("int")).alias("found"),
                                 F.round(F.avg(F.col("found").cast("int")), 3).alias("recall")).orderBy("what_matched").show(truncate=False))
(sec.filter("in_silver = 2").groupBy("name_variant")
    .agg(F.count("*").alias("people"), F.sum(F.col("found").cast("int")).alias("found"),
         F.round(F.avg(F.col("found").cast("int")), 3).alias("recall")).orderBy("name_variant").show(truncate=False))

In [ ]:
# ---- Does the review tier behave? ----------------------------------------------------
# Pairs left for review were NOT merged. How many of them were actually the same person?
pairs = spark.table("silver_identity_pairs")
tk = truth.select(F.col("client_id").alias("cid"), F.col("person_id").alias("pid"))
rv = (pairs.join(tk.withColumnRenamed("cid", "client_id_a").withColumnRenamed("pid", "pid_a"), "client_id_a", "left")
           .join(tk.withColumnRenamed("cid", "client_id_b").withColumnRenamed("pid", "pid_b"), "client_id_b", "left")
           .withColumn("same_person", F.col("pid_a") == F.col("pid_b")))
(rv.groupBy("match_rule", "decision").agg(F.count("*").alias("pairs"), F.sum(F.col("same_person").cast("int")).alias("really_same_person"))
   .orderBy(F.col("pairs").desc()).show(truncate=False))

In [ ]:
# ---- Misses: the real pairs the pipeline did not link --------------------------------
latest = spark.table("silver_crm_client").withColumn("rk", F.row_number().over(
    __import__("pyspark.sql", fromlist=["Window"]).Window.partitionBy("client_id").orderBy(F.col("updated_at").desc()))).filter("rk = 1")
missed = (two.filter("in_silver = 2 and keys = 2").select("person_id")
          .join(t, "person_id").select("person_id", "client_id", "name_variant", "email_same", "phone_same")
          .join(latest.select("client_id", "first_name", "last_name", "region"), "client_id"))
missed.orderBy("person_id", "client_id").show(20, truncate=False)

In [ ]:
# ---- How much of the miss does the review list catch? -------------------------------
# A missed person is "caught" when one of their pairs sits in the review list.
rev_real = (rv.filter("decision = 'review' and same_person").select("pid_a").distinct())
missed_people = missed.select("person_id").distinct()
caught = missed_people.join(rev_real, missed_people.person_id == rev_real.pid_a).count()
print("people missed by auto-linking:", missed_people.count(), "| of those, present in the review list:", caught)
top = rv.filter("decision = 'review' and match_rule = 'name_only_rare_name'")
print("rare-name review pairs:", top.count(), "| really the same person:", top.filter("same_person").count())

# ---- Save the headline numbers so the README and the dashboard can use them ----------
from pyspark.sql import Row
res = Row(eval_id="E" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S"), evaluated_at=datetime.now(timezone.utc).replace(tzinfo=None),
          people_with_two_records=two.count(), findable=findable.count(), found=found.count(),
          recall=round(found.count() / max(findable.count(), 1), 4),
          linked_groups=merged.count(), false_merges=false_merges,
          precision=round(1 - false_merges / max(merged.count(), 1), 4))
spark.createDataFrame([res]).write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("ctl_identity_eval")
spark.table("ctl_identity_eval").orderBy(F.col("evaluated_at").desc()).show(3, truncate=False)